# Fraud Detection Dataset Generation
Generate synthetic fraud detection dataset for MLOps monitoring

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import boto3
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

In [ ]:
def generate_fraud_dataset(n_samples=50000):
    # Create base features
    X, y = make_classification(
        n_samples=n_samples,
        n_features=20,
        n_informative=15,
        n_redundant=5,
        n_clusters_per_class=1,
        weights=[0.95, 0.05],  # 5% fraud rate
        random_state=42
    )
    
    # Create meaningful feature names
    feature_names = [
        'transaction_amount', 'account_age_days', 'num_transactions_day',
        'avg_transaction_amount', 'time_since_last_transaction',
        'merchant_category', 'payment_method', 'device_score',
        'location_risk_score', 'velocity_score', 'amount_deviation',
        'weekend_transaction', 'late_night_transaction', 'cross_border',
        'new_merchant', 'high_risk_country', 'suspicious_pattern',
        'account_verification_level', 'transaction_frequency', 'user_behavior_score'
    ]
    
    # Create DataFrame
    df = pd.DataFrame(X, columns=feature_names)
    df['is_fraud'] = y
    df['transaction_id'] = range(len(df))
    df['timestamp'] = pd.date_range(
        start='2024-01-01', 
        periods=len(df), 
        freq='5min'
    )
    
    return df

In [ ]:
# Generate and save dataset
df = generate_fraud_dataset()
print(f"Dataset shape: {df.shape}")
print(f"Fraud rate: {df['is_fraud'].mean():.3f}")
df.head()

In [ ]:
# Save locally first
df.to_csv('../data/raw/fraud_dataset.csv', index=False)
print("Dataset saved locally")

In [ ]:
# Upload to S3 (update bucket name)
bucket_name = 'your-mlops-monitoring-bucket'  # Replace with your bucket
s3_key = 'data/fraud_dataset.csv'

try:
    s3 = boto3.client('s3')
    s3.upload_file('../data/raw/fraud_dataset.csv', bucket_name, s3_key)
    print(f"Dataset uploaded to s3://{bucket_name}/{s3_key}")
except Exception as e:
    print(f"S3 upload failed: {e}")
    print("Dataset saved locally only")